# Food Detection — YOLOv8 Training
**Runtime → Change runtime type → T4 GPU** before running any cell.

This notebook: downloads datasets from Kaggle, converts to YOLO format, trains YOLOv8s, evaluates, and saves `best.pt` to your Google Drive.

In [ ]:
# Cell 1 — Check GPU & install packages
!nvidia-smi
!pip install ultralytics kaggle opencv-python-headless matplotlib pyyaml pandas -q
print('All packages installed.')

In [ ]:
# Cell 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_DIR = '/content/drive/MyDrive/FoodDetection'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive ready → {DRIVE_DIR}')

In [ ]:
# Cell 3 — Kaggle API setup
# 1. Go to kaggle.com → Account → API → Create New Token → downloads kaggle.json
# 2. Upload kaggle.json via Files panel (folder icon, left sidebar)
# 3. Run this cell
import shutil, os
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('/content/kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle API ready.')

In [ ]:
# Cell 4 — Download datasets (~1.5 GB total, takes 3-5 min)
os.makedirs('/content/datasets', exist_ok=True)
print('Downloading Indian Food Images...')
!kaggle datasets download -d iamsouravbanerjee/indian-food-images-dataset -p /content/datasets/indian --unzip -q
print('Downloading Food-101...')
!kaggle datasets download -d dansbecker/food-101 -p /content/datasets/food101 --unzip -q
print('Done.')

In [ ]:
# Cell 5 — Explore dataset structure
from pathlib import Path
indian_root = '/content/datasets/indian/Indian Food Images'
food101_root = '/content/datasets/food101/food-101/images'
indian_classes = sorted([d.name for d in Path(indian_root).iterdir() if d.is_dir()])
food101_classes = sorted([d.name for d in Path(food101_root).iterdir() if d.is_dir()])
print(f'Indian Food classes ({len(indian_classes)}): {indian_classes[:10]}...')
print(f'Food-101 classes ({len(food101_classes)}): {food101_classes[:10]}...')

In [ ]:
# Cell 6 — Merge datasets
import shutil, json
from pathlib import Path

MERGED_ROOT = '/content/merged_dataset'

# Select Indian classes (all of them)
INDIAN_CLASSES = sorted([d.name for d in Path(indian_root).iterdir() if d.is_dir()])

# Select complementary Food-101 classes
FOOD101_SELECTED = [
    'fried_rice','spring_rolls','dumplings','noodles',
    'grilled_salmon','omelette','pancakes','waffles',
    'pizza','hamburger','french_fries','caesar_salad',
]

def normalize(name):
    return name.lower().replace(' ', '_').replace('-', '_')

all_classes = []
os.makedirs(MERGED_ROOT, exist_ok=True)

for cls in INDIAN_CLASSES:
    src = Path(indian_root) / cls
    norm = normalize(cls)
    dst = Path(MERGED_ROOT) / norm
    dst.mkdir(exist_ok=True)
    imgs = list(src.glob('*.*'))[:300]
    for img in imgs: shutil.copy(img, dst / img.name)
    all_classes.append(norm)

for cls in FOOD101_SELECTED:
    src = Path(food101_root) / cls
    if not src.exists(): continue
    dst = Path(MERGED_ROOT) / cls
    dst.mkdir(exist_ok=True)
    imgs = list(src.glob('*.*'))[:300]
    for img in imgs: shutil.copy(img, dst / img.name)
    all_classes.append(cls)

all_classes = sorted(set(all_classes))
print(f'Total classes: {len(all_classes)}')
print(all_classes)

In [ ]:
# Cell 7 — Convert to YOLO format
import random, yaml

YOLO_ROOT = '/content/yolo_dataset'

for split in ('train','val','test'):
    Path(YOLO_ROOT,'images',split).mkdir(parents=True,exist_ok=True)
    Path(YOLO_ROOT,'labels',split).mkdir(parents=True,exist_ok=True)

class_to_id = {cls:i for i,cls in enumerate(all_classes)}
all_files = []
for cls in all_classes:
    for img in Path(MERGED_ROOT,cls).glob('*.*'):
        if img.suffix.lower() in {'.jpg','.jpeg','.png'}:
            all_files.append((img, cls))

random.shuffle(all_files)
n = len(all_files)
test_files  = all_files[:int(n*0.10)]
val_files   = all_files[int(n*0.10):int(n*0.25)]
train_files = all_files[int(n*0.25):]

def write_split(files, split):
    for img_path, cls in files:
        shutil.copy(img_path, Path(YOLO_ROOT,'images',split,img_path.name))
        lbl = Path(YOLO_ROOT,'labels',split,img_path.stem+'.txt')
        lbl.write_text(f'{class_to_id[cls]} 0.500000 0.500000 1.000000 1.000000\n')

write_split(train_files,'train')
write_split(val_files,'val')
write_split(test_files,'test')

# Write data.yaml
data_yaml = {'path':YOLO_ROOT,'train':'images/train','val':'images/val',
             'test':'images/test','nc':len(all_classes),'names':all_classes}
yaml_path = f'{YOLO_ROOT}/data.yaml'
with open(yaml_path,'w') as f: yaml.dump(data_yaml,f,default_flow_style=False)

# Save class names to Drive NOW (needed by Flask app)
with open(f'{DRIVE_DIR}/class_names.json','w') as f: json.dump(all_classes,f,indent=2)

print(f'Train:{len(train_files)} Val:{len(val_files)} Test:{len(test_files)}')
print(f'data.yaml saved → {yaml_path}')

In [ ]:
# Cell 8 — Train YOLOv8s  (≈ 30-50 min on T4 for 50 epochs)
from ultralytics import YOLO

model = YOLO('yolov8s.pt')   # auto-downloads COCO pretrained weights

results = model.train(
    data     = yaml_path,
    epochs   = 50,
    imgsz    = 640,
    batch    = 16,
    workers  = 2,
    patience = 10,
    device   = 0,
    project  = '/content/runs/train',
    name     = 'food_detector',
    exist_ok = True,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    fliplr=0.5, mosaic=1.0, mixup=0.1,
)
print('Training complete.')

In [ ]:
# Cell 9 — Evaluate on test set
best_pt = '/content/runs/train/food_detector/weights/best.pt'
eval_model = YOLO(best_pt)
metrics = eval_model.val(data=yaml_path, split='test')
print(f'mAP50    : {metrics.box.map50:.4f}')
print(f'mAP50-95 : {metrics.box.map:.4f}')
print(f'Precision: {metrics.box.p.mean():.4f}')
print(f'Recall   : {metrics.box.r.mean():.4f}')

In [ ]:
# Cell 10 — Plot training curves
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('/content/runs/train/food_detector/results.csv')
df.columns = df.columns.str.strip()
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(df['train/box_loss'],label='train'); axes[0].plot(df['val/box_loss'],label='val')
axes[0].set_title('Box Loss'); axes[0].legend()
axes[1].plot(df['train/cls_loss'],label='train'); axes[1].plot(df['val/cls_loss'],label='val')
axes[1].set_title('Class Loss'); axes[1].legend()
axes[2].plot(df['metrics/mAP50(B)'],label='mAP50')
axes[2].set_title('mAP50'); axes[2].legend()
plt.tight_layout()
plt.savefig(f'{DRIVE_DIR}/training_curves.png', dpi=150)
plt.show()

In [ ]:
# Cell 11 — Test on a sample plate image
# Upload your test image via Files panel first
import cv2, numpy as np
from matplotlib import pyplot as plt

TEST_IMAGE = '/content/test_plate.jpg'   # change path as needed
CONF_THRESH = 0.35

inf_model = YOLO(best_pt)
results = inf_model.predict(TEST_IMAGE, conf=CONF_THRESH, verbose=False)
boxes = results[0].boxes

img = cv2.cvtColor(cv2.imread(TEST_IMAGE), cv2.COLOR_BGR2RGB)
annotated = img.copy()

PALETTE = [(99,102,241),(20,184,166),(245,158,11),(239,68,68),(34,197,94)]

for i, box in enumerate(boxes):
    x1,y1,x2,y2 = map(int, box.xyxy[0].tolist())
    cls_id = int(box.cls[0])
    conf   = float(box.conf[0])
    name   = all_classes[cls_id].replace('_',' ').title()
    color  = PALETTE[i % len(PALETTE)]
    cx,cy  = (x1+x2)//2, (y1+y2)//2
    radius = max(x2-x1, y2-y1)//2 + 10
    cv2.circle(annotated, (cx,cy), radius, color, 3)
    label = f'{name} {conf*100:.0f}%'
    cv2.putText(annotated, label, (cx-60, cy-radius-8),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

plt.figure(figsize=(10,8))
plt.imshow(annotated)
plt.axis('off')
plt.title(f'Detected {len(boxes)} food item(s)')
plt.show()

In [ ]:
# Cell 12 — Save model to Google Drive
shutil.copy(best_pt, f'{DRIVE_DIR}/best.pt')
print(f'Saved best.pt → {DRIVE_DIR}/best.pt')
print()
print('Next steps:')
print('  1. Download best.pt from Drive')
print('  2. Download class_names.json from Drive')
print('  3. Place both in flask_app/models/')
print('  4. Run the Flask app locally')